<a href="https://colab.research.google.com/github/adg1205/CSE425-Project/blob/master/Hard%20Task/Hard_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install librosa soundfile scikit-learn umap-learn matplotlib tqdm sentence-transformers

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
AUDIO_ROOT  = "/content/drive/MyDrive/Project Dataset/genres_original"
LYRICS_ROOT = "/content/drive/MyDrive/Project Dataset/lyrics_generated"

OUT_DIR     = "/content/drive/MyDrive/hard_task_results"
os.makedirs(OUT_DIR, exist_ok=True)
LATVIS_DIR  = os.path.join(OUT_DIR, "latent_visualization")
RECON_DIR   = os.path.join(OUT_DIR, "reconstructions")
os.makedirs(LATVIS_DIR, exist_ok=True)
os.makedirs(RECON_DIR, exist_ok=True)

print("AUDIO_ROOT exists?", os.path.isdir(AUDIO_ROOT), AUDIO_ROOT)
print("LYRICS_ROOT exists?", os.path.isdir(LYRICS_ROOT), LYRICS_ROOT)
print("OUT_DIR:", OUT_DIR)

AUDIO_ROOT exists? True /content/drive/MyDrive/Project Dataset/genres_original
LYRICS_ROOT exists? True /content/drive/MyDrive/Project Dataset/lyrics_generated
OUT_DIR: /content/drive/MyDrive/hard_task_results


In [ ]:
#Build Paired Index (Audio & Lyrics) + Labels
import glob
import numpy as np
import pandas as pd

def build_paired_index(audio_root, lyrics_root, max_files_per_genre=None, max_files_total=None):
    genre_dirs = sorted([d for d in glob.glob(os.path.join(audio_root, "*")) if os.path.isdir(d)])
    rows = []
    for gdir in genre_dirs:
        genre = os.path.basename(gdir)
        wavs = sorted(glob.glob(os.path.join(gdir, "*.wav")))
        if max_files_per_genre is not None:
            wavs = wavs[:max_files_per_genre]
        for wav in wavs:
            track_id = os.path.splitext(os.path.basename(wav))[0]
            lpath = os.path.join(lyrics_root, genre, f"{track_id}.txt")
            rows.append((genre, track_id, wav, lpath if os.path.exists(lpath) else None))

    df = pd.DataFrame(rows, columns=["genre","track_id","audio_path","lyrics_path"])
    if max_files_total is not None and len(df) > max_files_total:
        df = df.sample(n=max_files_total, random_state=42).reset_index(drop=True)

    coverage = df["lyrics_path"].notna().mean() * 100
    print(f"Tracks={len(df)} | Genres={df['genre'].nunique()} | Lyrics coverage={coverage:.1f}%")
    return df

# Debug fast first; set to None for full run
MAX_FILES_PER_GENRE = None  # e.g., 20 for quick test
MAX_FILES_TOTAL = None      # e.g., 200 for quick test

df = build_paired_index(AUDIO_ROOT, LYRICS_ROOT, MAX_FILES_PER_GENRE, MAX_FILES_TOTAL)

genres = sorted(df["genre"].unique().tolist())
genre_to_id = {g:i for i,g in enumerate(genres)}
id_to_genre = {i:g for g,i in genre_to_id.items()}
Y = np.array([genre_to_id[g] for g in df["genre"]], dtype=np.int64)

df.to_csv(os.path.join(OUT_DIR, "paired_index.csv"), index=False)
df.head()

Tracks=700 | Genres=7 | Lyrics coverage=100.0%


,genre,track_id,audio_path,lyrics_path
0,blues,blues.00000,/content/drive/MyDrive/Project Dataset/genres_...,/content/drive/MyDrive/Project Dataset/lyrics_...
1,blues,blues.00001,/content/drive/MyDrive/Project Dataset/genres_...,/content/drive/MyDrive/Project Dataset/lyrics_...
2,blues,blues.00002,/content/drive/MyDrive/Project Dataset/genres_...,/content/drive/MyDrive/Project Dataset/lyrics_...
3,blues,blues.00003,/content/drive/MyDrive/Project Dataset/genres_...,/content/drive/MyDrive/Project Dataset/lyrics_...
4,blues,blues.00004,/content/drive/MyDrive/Project Dataset/genres_...,/content/drive/MyDrive/Project Dataset/lyrics_...


In [ ]:
# Audio features (fixed-size log-mel spectograms)
import librosa

SR = 22050
DURATION = 30.0
N_MELS = 64
N_FFT = 1024
HOP = 512

TARGET_SAMPLES = int(SR * DURATION)
TARGET_FRAMES = 1 + TARGET_SAMPLES // HOP

def logmel_fixed(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    if len(y) < TARGET_SAMPLES:
        y = np.pad(y, (0, TARGET_SAMPLES - len(y)))
    else:
        y = y[:TARGET_SAMPLES]

    mel = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS, power=2.0
    )
    logmel = librosa.power_to_db(mel, ref=np.max).astype(np.float32)  # (M, T)

    if logmel.shape[1] < TARGET_FRAMES:
        logmel = np.pad(logmel, ((0,0),(0, TARGET_FRAMES-logmel.shape[1])), mode="constant")
    else:
        logmel = logmel[:, :TARGET_FRAMES]
    return logmel

def estimate_audio_norm(df, max_items=300):
    idxs = np.arange(len(df))
    if len(df) > max_items:
        np.random.seed(42)
        idxs = np.random.choice(idxs, size=max_items, replace=False)
    mats = np.stack([logmel_fixed(df.loc[i, "audio_path"]) for i in idxs], axis=0)
    return float(mats.mean()), float(mats.std() + 1e-6)

AUDIO_MEAN, AUDIO_STD = estimate_audio_norm(df, max_items=300)
print("Audio mean/std:", AUDIO_MEAN, AUDIO_STD)

Audio mean/std: -38.717933654785156 14.694234848022461


In [ ]:
# Lyrics Embeddings (Sentence Transformer)
from sentence_transformers import SentenceTransformer

TEXT_MODEL_NAME = "all-MiniLM-L6-v2"
text_model = SentenceTransformer(TEXT_MODEL_NAME)

def read_text(path):
    if path is None:
        return ""
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

texts = [read_text(p) for p in df["lyrics_path"].tolist()]
E = text_model.encode(texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True).astype(np.float32)

np.save(os.path.join(OUT_DIR, "lyrics_embeddings.npy"), E)
print("Lyrics embeddings:", E.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Lyrics embeddings: (700, 384)


In [ ]:
# Dataset/Dataloader
import torch
from torch.utils.data import Dataset, DataLoader

class HardDataset(Dataset):
    def __init__(self, df, E, Y, audio_mean, audio_std):
        self.df = df.reset_index(drop=True)
        self.E = E
        self.Y = Y
        self.audio_mean = audio_mean
        self.audio_std = audio_std

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        x = logmel_fixed(self.df.loc[i, "audio_path"])
        x = (x - self.audio_mean) / self.audio_std
        x = torch.tensor(x[None, :, :], dtype=torch.float32)  # (1,M,T)

        e = torch.tensor(self.E[i], dtype=torch.float32)
        y = torch.tensor(self.Y[i], dtype=torch.long)
        return x, e, y

ds = HardDataset(df, E, Y, AUDIO_MEAN, AUDIO_STD)
dl = DataLoader(ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

DEVICE: cuda


In [ ]:
# CVAE model (audio + lyrics + genre) with Beta-VAE loss
import torch.nn as nn
import torch.nn.functional as F

class ConvEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(),
        )
    def forward(self, x): return self.net(x)

class ConvDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 4, stride=2, padding=1),
        )
    def forward(self, h): return self.net(h)

class CVAE_MultiModal(nn.Module):
    def __init__(self, n_genres, lyrics_dim, n_mels, t_frames, latent_dim=16, cond_dim=32):
        super().__init__()
        self.latent_dim = latent_dim
        self.cond_dim = cond_dim

        self.genre_emb = nn.Embedding(n_genres, cond_dim)

        self.enc_conv = ConvEncoder()
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_mels, t_frames)
            h = self.enc_conv(dummy)
            self.h_shape = h.shape[1:]
            self.h_dim = int(np.prod(self.h_shape))

        self.audio_fc = nn.Linear(self.h_dim, 256)

        self.lyrics_mlp = nn.Sequential(
            nn.Linear(lyrics_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
        )

        self.fuse = nn.Sequential(
            nn.Linear(256 + 128 + cond_dim, 256), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)

        # Decoder: reconstruct audio
        self.dec_fc = nn.Sequential(
            nn.Linear(latent_dim + cond_dim, 256), nn.ReLU(),
            nn.Linear(256, self.h_dim), nn.ReLU(),
        )
        self.dec_conv = ConvDecoder()

        # Decoder head: predict lyrics embedding (multi-modal reconstruction)
        self.lyrics_head = nn.Sequential(
            nn.Linear(latent_dim + cond_dim, 256), nn.ReLU(),
            nn.Linear(256, lyrics_dim),
        )

    def encode(self, x, e, y):
        h = self.enc_conv(x).view(x.size(0), -1)
        a = torch.relu(self.audio_fc(h))
        l = self.lyrics_mlp(e)
        c = self.genre_emb(y)
        f = self.fuse(torch.cat([a, l, c], dim=1))
        return self.fc_mu(f), self.fc_logvar(f)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, y):
        c = self.genre_emb(y)
        zy = torch.cat([z, c], dim=1)

        hflat = self.dec_fc(zy)
        h = hflat.view(z.size(0), *self.h_shape)
        x_hat = self.dec_conv(h)

        e_hat = self.lyrics_head(zy)
        return x_hat, e_hat

    def forward(self, x, e, y):
        mu, logvar = self.encode(x, e, y)
        z = self.reparameterize(mu, logvar)
        x_hat, e_hat = self.decode(z, y)
        return x_hat, e_hat, mu, logvar

def cvae_loss(x, x_hat, e, e_hat, mu, logvar, beta=4.0, lambda_lyrics=0.2):
    # crop if transpose conv overshoots
    x_hat = x_hat[:, :, :x.shape[2], :x.shape[3]]

    recon_audio = F.mse_loss(x_hat, x, reduction="mean")
    recon_lyrics = F.mse_loss(e_hat, e, reduction="mean")
    kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    loss = recon_audio + lambda_lyrics * recon_lyrics + beta * kld
    return loss, recon_audio, recon_lyrics, kld

In [ ]:
#Train CVAE (Beta-VAE style) + extract latents
import torch
from tqdm import tqdm

model = CVAE_MultiModal(
    n_genres=len(genres),
    lyrics_dim=E.shape[1],
    n_mels=N_MELS,
    t_frames=TARGET_FRAMES,
    latent_dim=16,
    cond_dim=32
).to(DEVICE)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 50
BETA = 3.0           # beta>1 encourages more disentanglement
LAMBDA_LYRICS = 0.2  # weight for lyrics reconstruction

history = []
model.train()
for ep in range(1, EPOCHS+1):
    tot = ra = rl = kld = 0.0
    n = 0
    for xb, eb, yb in dl:
        xb, eb, yb = xb.to(DEVICE), eb.to(DEVICE), yb.to(DEVICE)

        x_hat, e_hat, mu, logvar = model(xb, eb, yb)
        loss, recon_a, recon_l, k = cvae_loss(
            xb, x_hat, eb, e_hat, mu, logvar,
            beta=BETA, lambda_lyrics=LAMBDA_LYRICS
        )

        opt.zero_grad()
        loss.backward()
        opt.step()

        bs = xb.size(0)
        n += bs
        tot += loss.item() * bs
        ra += recon_a.item() * bs
        rl += recon_l.item() * bs
        kld += k.item() * bs

    history.append({
        "epoch": ep,
        "loss": tot/n,
        "recon_audio": ra/n,
        "recon_lyrics": rl/n,
        "kld": kld/n
    })
    if ep % 10 == 0 or ep == 1:
        print(f"Epoch {ep:02d} | loss={tot/n:.4f} ra={ra/n:.4f} rl={rl/n:.4f} kld={kld/n:.4f}")

pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "cvae_train_history.csv"), index=False)

# Latent extraction (use mu)
model.eval()
Z = []
with torch.no_grad():
    for xb, eb, yb in DataLoader(ds, batch_size=32, shuffle=False):
        mu, _ = model.encode(xb.to(DEVICE), eb.to(DEVICE), yb.to(DEVICE))
        Z.append(mu.cpu().numpy())
Z = np.concatenate(Z, axis=0)
np.save(os.path.join(OUT_DIR, "latent_mu.npy"), Z)
print("Z:", Z.shape)

Epoch 01 | loss=0.7338 ra=0.7292 rl=0.0141 kld=0.0006
Epoch 10 | loss=0.4866 ra=0.4749 rl=0.0023 kld=0.0038
Epoch 20 | loss=0.4818 ra=0.4656 rl=0.0021 kld=0.0053
Epoch 30 | loss=0.4792 ra=0.4622 rl=0.0020 kld=0.0055
Epoch 40 | loss=0.4670 ra=0.4532 rl=0.0020 kld=0.0045
Epoch 50 | loss=0.4746 ra=0.4574 rl=0.0020 kld=0.0056
Z: (700, 16)


In [ ]:
#Clustering + Hard-task metrics (Silhouette, NMI, ARI, Purity)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

def cluster_purity(y_true, y_pred):
    # purity = sum_k max_j |Ck ∩ Tj| / N
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    total = 0
    for c in np.unique(y_pred):
        idx = np.where(y_pred == c)[0]
        if len(idx) == 0:
            continue
        counts = np.bincount(y_true[idx])
        total += counts.max()
    return total / len(y_true)

Zs = StandardScaler().fit_transform(Z)
k = len(genres)

pred = KMeans(n_clusters=k, random_state=42, n_init="auto").fit_predict(Zs)

metrics = {
    "method": "CVAE(beta)+KMeans",
    "k": k,
    "silhouette": float(silhouette_score(Zs, pred)),
    "NMI": float(normalized_mutual_info_score(Y, pred)),
    "ARI": float(adjusted_rand_score(Y, pred)),
    "purity": float(cluster_purity(Y, pred)),
}
metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(os.path.join(OUT_DIR, "hard_metrics_cvae.csv"), index=False)
metrics_df

,method,k,silhouette,NMI,ARI,purity
0,CVAE(beta)+KMeans,7,0.523956,0.374076,0.15887,0.39


In [ ]:
#Baselines required in Hard task (PCA+KMeans, AE+KMeans, direct spectral)
from sklearn.decomposition import PCA

# Direct spectral feature clustering: pooled log-mel only
def audio_pooled_logmel(df):
    feats = []
    for p in df["audio_path"].tolist():
        X = logmel_fixed(p)  # (M,T)
        feats.append(np.concatenate([X.mean(axis=1), X.std(axis=1)], axis=0))  # (2M,)
    return np.stack(feats, axis=0).astype(np.float32)

A_pool = audio_pooled_logmel(df)
E_lyr = E.astype(np.float32)

A_std = StandardScaler().fit_transform(A_pool)
E_std = StandardScaler().fit_transform(E_lyr)
H = np.concatenate([A_std, E_std], axis=1)  # simple hybrid vector for baselines

# Baseline 1: PCA + KMeans
H_pca = PCA(n_components=Z.shape[1], random_state=42).fit_transform(H)
H_pca_s = StandardScaler().fit_transform(H_pca)
pred_pca = KMeans(n_clusters=k, random_state=42, n_init="auto").fit_predict(H_pca_s)

# Baseline 2: Autoencoder + KMeans (MLP AE on H)
import torch.nn as nn
import torch.nn.functional as F
import torch

class MLP_AE(nn.Module):
    def __init__(self, input_dim, latent_dim=16, hidden=256):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, latent_dim))
        self.dec = nn.Sequential(nn.Linear(latent_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, input_dim))
    def forward(self, x):
        z = self.enc(x)
        xh = self.dec(z)
        return xh, z

Xh = torch.tensor(H, dtype=torch.float32)
ae = MLP_AE(input_dim=H.shape[1], latent_dim=Z.shape[1]).to(DEVICE)
opt_ae = torch.optim.Adam(ae.parameters(), lr=1e-3)

ae.train()
for ep in range(1, 51):
    idx = torch.randperm(Xh.size(0))
    for i in range(0, Xh.size(0), 256):
        xb = Xh[idx[i:i+256]].to(DEVICE)
        xhat, z = ae(xb)
        loss = F.mse_loss(xhat, xb)
        opt_ae.zero_grad()
        loss.backward()
        opt_ae.step()
    if ep % 10 == 0 or ep == 1:
        print("AE epoch", ep, "loss", float(loss.detach().cpu()))

ae.eval()
with torch.no_grad():
    _, Z_ae = ae(Xh.to(DEVICE))
Z_ae = Z_ae.cpu().numpy()
Z_ae_s = StandardScaler().fit_transform(Z_ae)
pred_ae = KMeans(n_clusters=k, random_state=42, n_init="auto").fit_predict(Z_ae_s)

# Baseline 3: direct spectral features clustering (no PCA)
A_s = StandardScaler().fit_transform(A_pool)
pred_spec = KMeans(n_clusters=k, random_state=42, n_init="auto").fit_predict(A_s)

def metric_pack(name, X, labels):
    return {
        "method": name,
        "silhouette": float(silhouette_score(X, labels)),
        "NMI": float(normalized_mutual_info_score(Y, labels)),
        "ARI": float(adjusted_rand_score(Y, labels)),
        "purity": float(cluster_purity(Y, labels))
    }

baseline_rows = [
    metric_pack("BASELINE: PCA+KMeans", H_pca_s, pred_pca),
    metric_pack("BASELINE: AE+KMeans", Z_ae_s, pred_ae),
    metric_pack("BASELINE: DirectSpectral(KMeans)", A_s, pred_spec),
    metric_pack("MODEL: CVAE(beta)+KMeans", Zs, pred),
]
all_metrics_df = pd.DataFrame(baseline_rows)
all_metrics_df.to_csv(os.path.join(OUT_DIR, "hard_metrics_all_methods.csv"), index=False)
all_metrics_df

AE epoch 1 loss 0.9972813129425049
AE epoch 10 loss 0.6781168580055237
AE epoch 20 loss 0.5846936702728271
AE epoch 30 loss 0.5182310938835144
AE epoch 40 loss 0.48999443650245667
AE epoch 50 loss 0.47896939516067505


,method,silhouette,NMI,ARI,purity
0,BASELINE: PCA+KMeans,0.100293,0.117137,0.079522,0.311429
1,BASELINE: AE+KMeans,0.110685,0.141707,0.097618,0.347143
2,BASELINE: DirectSpectral(KMeans),0.152641,0.219358,0.118533,0.394286
3,MODEL: CVAE(beta)+KMeans,0.523956,0.374076,0.158870,0.390000


In [ ]:
# Required visualizations (latent plots, cluster distribution, reconstructions)
import umap
import matplotlib.pyplot as plt

# UMAP latent plots
Z2 = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(Zs)

plt.figure(figsize=(7,6))
plt.scatter(Z2[:,0], Z2[:,1], c=pred, s=10, cmap="tab10")
plt.title("UMAP of CVAE latent (colored by predicted cluster)")
plt.tight_layout()
plt.savefig(os.path.join(LATVIS_DIR, "umap_latent_by_cluster.png"), dpi=200)
plt.close()

plt.figure(figsize=(7,6))
plt.scatter(Z2[:,0], Z2[:,1], c=Y, s=10, cmap="tab10")
plt.title("UMAP of CVAE latent (colored by true genre)")
plt.tight_layout()
plt.savefig(os.path.join(LATVIS_DIR, "umap_latent_by_genre.png"), dpi=200)
plt.close()

# Cluster distribution over genres (heatmap-like table plot)
ct = pd.crosstab(pd.Series(pred, name="cluster"), pd.Series(Y, name="genre_id"))
plt.figure(figsize=(10,4))
plt.imshow(ct.values, aspect="auto")
plt.yticks(range(ct.shape[0]), ct.index.tolist())
plt.xticks(range(ct.shape[1]), [id_to_genre[i] for i in ct.columns.tolist()], rotation=45, ha="right")
plt.colorbar(label="count")
plt.title("Cluster distribution over genres")
plt.tight_layout()
plt.savefig(os.path.join(LATVIS_DIR, "cluster_genre_distribution.png"), dpi=200)
plt.close()

print("Saved plots to:", LATVIS_DIR)

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved plots to: /content/drive/MyDrive/hard_task_results/latent_visualization
